In [ ]:
import os
import sys
import json
import re
import shutil
import subprocess
import tempfile
import urllib.request
from pathlib import Path

WORK = Path("/kaggle/working/webgpu-browser")
WORK.mkdir(parents=True, exist_ok=True)

def run(*args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)

def identify_gpu():
    """Require a working T4 session before installing browser dependencies."""
    nvidia_smi = shutil.which("nvidia-smi")
    if nvidia_smi is None:
        # Some NVIDIA images expose the utility outside the default PATH.
        nvidia_smi = next((
            str(candidate)
            for candidate in (
                Path("/usr/bin/nvidia-smi"),
                Path("/usr/local/nvidia/bin/nvidia-smi"),
                Path("/opt/bin/nvidia-smi"),
            )
            if candidate.is_file() and os.access(candidate, os.X_OK)
        ), None)

    if nvidia_smi is None:
        raise RuntimeError(
            "NVIDIA GPU tools are unavailable (nvidia-smi not found). "
            "This session may be running on CPU or TPU. In Kaggle, select "
            "Settings > Accelerator > GPU T4, enable Internet, and let the "
            "session restart before rerunning Cell 1. If GPU selection is "
            "unavailable, check your account's GPU access and remaining quota. "
            "Do not install NVIDIA host drivers to fix a missing GPU session."
        )

    try:
        result = subprocess.check_output(
            [nvidia_smi, "--query-gpu=name,driver_version", "--format=csv,noheader"],
            text=True,
            stderr=subprocess.STDOUT,
            timeout=30,
        )
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
        raise RuntimeError(
            "nvidia-smi exists but could not query the GPU. Restart the Kaggle "
            "session with GPU T4 selected, then rerun Cell 1."
        ) from error

    print("Detected NVIDIA GPUs:\n" + result.strip())
    for line in result.splitlines():
        name, separator, version = line.rpartition(",")
        if separator and re.search(r"\bT4\b", name, re.IGNORECASE):
            driver = version.strip()
            if re.fullmatch(r"\d+(?:\.\d+)+", driver):
                return driver

    raise RuntimeError(
        "This notebook requires an NVIDIA T4 with a valid driver version. "
        "Select GPU T4 in Kaggle's accelerator settings and restart the session."
    )


# Identify the host driver; do not replace its kernel/compute packages.
driver = identify_gpu()

branch = driver.split(".")[0]
print("Host NVIDIA driver:", driver)

# Browser and general system dependencies.
run(sys.executable, "-m", "pip", "install", "-q", "playwright", "aiohttp")
run(sys.executable, "-m", "playwright", "install", "--with-deps", "chromium")

run("apt-get", "update", "-qq")
run("apt-get", "install", "-y", "vulkan-tools", "libvulkan1")

# Download and extract NVIDIA graphics libraries without installing them.
private_root = Path(tempfile.mkdtemp(prefix="nvidia-", dir=str(WORK)))
downloads = private_root / "downloads"
libraries = private_root / "libraries"
downloads.mkdir()
libraries.mkdir()

for package in [
    f"libnvidia-gl-{branch}",
    f"libnvidia-gpucomp-{branch}",
]:
    listing = subprocess.check_output(
        ["apt-cache", "madison", package],
        text=True
    )

    versions = [
        line.split("|")[1].strip()
        for line in listing.splitlines()
        if "|" in line
    ]

    version = next(
        (
            v for v in versions
            if v.split(":")[-1].startswith(driver + "-")
        ),
        None
    )

    if version is None:
        raise RuntimeError(
            f"No {package} package matching host driver {driver}.\n"
            f"Available versions:\n{listing}"
        )

    print("Downloading:", package, version)
    run(
        "apt-get", "download", f"{package}:amd64={version}",
        cwd=str(downloads)
    )

for archive in downloads.glob("*.deb"):
    run("dpkg-deb", "--extract", str(archive), str(libraries))

# IMPORTANT: use standard Vulkan through EGL, never Vulkan SC.
egl_files = [
    p for p in libraries.rglob(f"libEGL_nvidia.so.{driver}")
    if p.is_file()
]
if not egl_files:
    raise RuntimeError("Matching libEGL_nvidia was not extracted")

egl_library = egl_files[0].resolve()

manifest = None
for source in sorted(libraries.rglob("nvidia_icd.json")):
    candidate = json.loads(source.read_text())
    library_name = Path(
        candidate.get("ICD", {}).get("library_path", "")
    ).name

    if library_name.startswith(("libGLX_nvidia.so", "libEGL_nvidia.so")):
        manifest = candidate
        break

if manifest is None:
    raise RuntimeError("Standard NVIDIA Vulkan manifest was not extracted")

manifest["ICD"]["library_path"] = str(egl_library)
private_manifest = private_root / "nvidia-egl-vulkan.json"
private_manifest.write_text(json.dumps(manifest, indent=2))

# Only child processes receiving GPU_ENV use these private libraries.
GPU_ENV = os.environ.copy()

library_dirs = sorted({
    str(p.parent)
    for p in libraries.rglob("*.so*")
    if p.is_file()
})

existing_paths = GPU_ENV.get("LD_LIBRARY_PATH", "")
GPU_ENV["LD_LIBRARY_PATH"] = ":".join(
    library_dirs + ([existing_paths] if existing_paths else [])
)
GPU_ENV["VK_DRIVER_FILES"] = str(private_manifest)
GPU_ENV["VK_ICD_FILENAMES"] = str(private_manifest)

print("Vulkan driver:", egl_library)

result = subprocess.run(
    ["vulkaninfo", "--summary"],
    env=GPU_ENV,
    capture_output=True,
    text=True,
    timeout=60
)

output = result.stdout + result.stderr
(WORK / "vulkan.txt").write_text(output)
print(output)

if result.returncode != 0 or not re.search(
    r"deviceName\s*=.*T4", output, re.IGNORECASE
):
    raise RuntimeError(
        "T4 Vulkan verification failed. Inspect the output above."
    )

# Temporary remote-access tunnel.
CLOUDFLARED = WORK / "cloudflared"
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64",
        CLOUDFLARED
    )
    CLOUDFLARED.chmod(0o700)

print("\nPASS: T4 Vulkan works through EGL. Run Cell 2.")

In [ ]:
import asyncio
import contextlib
import json
import re
import secrets
import tempfile
import time
from pathlib import Path
from urllib.parse import urlsplit

from aiohttp import web, ClientSession, ClientTimeout, WSMsgType
from playwright.async_api import async_playwright


GPU_TEST = r"""
async () => {
  if (!navigator.gpu)
    throw Error("navigator.gpu unavailable");

  const adapter = await navigator.gpu.requestAdapter({
    powerPreference: "high-performance"
  });

  if (!adapter)
    throw Error("No WebGPU adapter");

  const info = adapter.info || await adapter.requestAdapterInfo();

  const details = {
    vendor: info.vendor,
    architecture: info.architecture,
    device: info.device,
    description: info.description
  };

  if (
    adapter.isFallbackAdapter ||
    info.isFallbackAdapter ||
    /swiftshader|llvmpipe|lavapipe|software/i.test(JSON.stringify(details))
  ) {
    throw Error("Software GPU rejected: " + JSON.stringify(details));
  }

  if (!/nvidia|0x10de/i.test(JSON.stringify(details))) {
    throw Error("Cannot confirm NVIDIA WebGPU: " + JSON.stringify(details));
  }

  const device = await adapter.requestDevice();

  try {
    device.pushErrorScope("validation");

    const output = device.createBuffer({
      size: 16,
      usage: GPUBufferUsage.STORAGE | GPUBufferUsage.COPY_SRC
    });

    const readback = device.createBuffer({
      size: 16,
      usage: GPUBufferUsage.COPY_DST | GPUBufferUsage.MAP_READ
    });

    const pipeline = await device.createComputePipelineAsync({
      layout: "auto",
      compute: {
        module: device.createShaderModule({
          code: `
            @group(0) @binding(0)
            var<storage, read_write> result: array<u32>;

            @compute @workgroup_size(4)
            fn main(@builtin(global_invocation_id) id: vec3u) {
              result[id.x] = (id.x + 1u) * 2u;
            }
          `
        }),
        entryPoint: "main"
      }
    });

    const bindings = device.createBindGroup({
      layout: pipeline.getBindGroupLayout(0),
      entries: [
        {binding: 0, resource: {buffer: output}}
      ]
    });

    const encoder = device.createCommandEncoder();
    const pass = encoder.beginComputePass();

    pass.setPipeline(pipeline);
    pass.setBindGroup(0, bindings);
    pass.dispatchWorkgroups(1);
    pass.end();

    encoder.copyBufferToBuffer(output, 0, readback, 0, 16);
    device.queue.submit([encoder.finish()]);

    await readback.mapAsync(GPUMapMode.READ);
    const values = Array.from(
      new Uint32Array(readback.getMappedRange())
    );
    readback.unmap();

    const error = await device.popErrorScope();
    if (error)
      throw Error(error.message);

    if (values.join() !== "2,4,6,8")
      throw Error("GPU compute verification failed");

    return {
      adapter: details,
      computeResult: values,
      features: Array.from(adapter.features)
    };
  } finally {
    device.destroy();
  }
}
"""


def is_nvidia_t4(device):
    # ANGLE may report vendorId=0 while identifying NVIDIA in strings.
    identity = " ".join(
        str(device.get(key, ""))
        for key in ("vendorString", "deviceString", "driverVendor")
    )

    software = re.search(
        r"swiftshader|llvmpipe|lavapipe|software",
        identity,
        re.IGNORECASE,
    )
    nvidia = (
        device.get("vendorId") == 0x10DE
        or "nvidia" in identity.lower()
    )
    t4 = re.search(r"\bT4\b", identity, re.IGNORECASE)

    return nvidia and bool(t4) and not software


async def capture_gpu_diagnostics(browser, page, work):
    print("\n=== CHROMIUM GPU DIAGNOSTICS ===")

    try:
        cdp = await browser.new_browser_cdp_session()
        try:
            system = await cdp.send("SystemInfo.getInfo")
        finally:
            await cdp.detach()

        (work / "chrome-system-info.json").write_text(
            json.dumps(system, indent=2)
        )
        print(json.dumps(system.get("gpu", system), indent=2))
    except Exception as error:
        print("SystemInfo diagnostic failed:", error)

    try:
        await page.goto(
            "chrome://gpu",
            wait_until="domcontentloaded",
            timeout=15000,
        )
        await page.wait_for_timeout(2000)

        # Collect text from normal DOM and nested shadow DOM.
        gpu_text = await page.evaluate(r"""
            () => {
              function collect(root) {
                const parts = [];
                for (const node of root.childNodes) {
                  if (node.nodeType === Node.TEXT_NODE) {
                    const text = node.textContent.trim();
                    if (text) parts.push(text);
                  } else if (node.nodeType === Node.ELEMENT_NODE) {
                    if (["SCRIPT", "STYLE"].includes(node.tagName))
                      continue;
                    parts.push(collect(node.shadowRoot || node));
                  }
                }
                return parts.filter(Boolean).join("\n");
              }
              return collect(document.body);
            }
        """)

        (work / "chrome-gpu.txt").write_text(gpu_text)
        print("\n=== CHROME://GPU ===")
        print(gpu_text[-24000:])
    except Exception as error:
        print("chrome://gpu diagnostic failed:", error)

    chrome_log = work / "chrome.log"
    if chrome_log.exists():
        print("\n=== CHROME LOG ===")
        print(chrome_log.read_text(errors="replace")[-16000:])


async def remote_browser(minutes=120):
    if not 1 <= minutes <= 720:
        raise ValueError("minutes must be between 1 and 720")

    if "GPU_ENV" not in globals():
        raise RuntimeError("Run Cell 1 successfully first.")

    work = Path("/kaggle/working/webgpu-browser")
    work.mkdir(parents=True, exist_ok=True)

    cloudflared = work / "cloudflared"
    if not cloudflared.is_file():
        raise RuntimeError("cloudflared missing. Finish Cell 1 first.")

    token = secrets.token_urlsafe(32)
    children = []
    logs = []
    runner = None

    profile = tempfile.TemporaryDirectory(
        prefix="chrome-", dir=str(work)
    )

    # Give Chromium a valid private runtime directory.
    runtime_dir = Path(profile.name) / "runtime"
    runtime_dir.mkdir(mode=0o700)

    browser_env = GPU_ENV.copy()
    browser_env["XDG_RUNTIME_DIR"] = str(runtime_dir)

    try:
        async with async_playwright() as pw:
            async with ClientSession(
                timeout=ClientTimeout(total=None, sock_connect=15)
            ) as http:
                try:
                    chrome_log = open(work / "chrome.log", "wb")
                    logs.append(chrome_log)

                    chrome = await asyncio.create_subprocess_exec(
                        pw.chromium.executable_path,
                        "--headless=new",
                        "--no-sandbox",
                        "--disable-dev-shm-usage",
                        "--enable-unsafe-webgpu",
                        "--use-angle=vulkan",
                        "--enable-features=Vulkan,VulkanFromANGLE,DefaultANGLEVulkan",
                        "--disable-vulkan-surface",
                        "--disable-software-rasterizer",
                        "--ignore-gpu-blocklist",
                        "--enable-logging=stderr",
                        "--v=1",
                        "--remote-debugging-address=127.0.0.1",
                        "--remote-debugging-port=9222",
                        "--user-data-dir=" + profile.name,
                        "about:blank",
                        env=browser_env,
                        stdout=chrome_log,
                        stderr=chrome_log,
                    )
                    children.append(chrome)

                    version = None
                    for _ in range(120):
                        if chrome.returncode is not None:
                            raise RuntimeError(
                                "Chrome exited. Inspect " + str(work / "chrome.log")
                            )
                        try:
                            async with http.get(
                                "http://127.0.0.1:9222/json/version",
                                timeout=ClientTimeout(total=2),
                            ) as response:
                                response.raise_for_status()
                                version = await response.json()
                            break
                        except Exception:
                            await asyncio.sleep(0.5)

                    if version is None:
                        raise RuntimeError(
                            "Chrome startup timed out. Inspect chrome.log"
                        )

                    browser = await pw.chromium.connect_over_cdp(
                        "http://127.0.0.1:9222"
                    )
                    print("Chromium version:", browser.version)

                    page = await browser.contexts[0].new_page()
                    page.on(
                        "console",
                        lambda msg: print(
                            f"[GPU probe {msg.type}] {msg.text}"
                        ),
                    )

                    try:
                        # Intercepted locally; localhost is a secure context.
                        await page.route(
                            "http://localhost/gpu-test",
                            lambda route: route.fulfill(
                                content_type="text/html",
                                body="<!doctype html><title>GPU verification</title>",
                            ),
                        )
                        await page.goto("http://localhost/gpu-test")

                        report = await asyncio.wait_for(
                            page.evaluate(GPU_TEST),
                            timeout=90,
                        )

                        cdp = await browser.new_browser_cdp_session()
                        try:
                            system = await cdp.send("SystemInfo.getInfo")
                        finally:
                            await cdp.detach()

                        devices = system["gpu"]["devices"]
                        if not any(is_nvidia_t4(d) for d in devices):
                            raise RuntimeError(
                                "Chrome did not confirm NVIDIA T4:\n"
                                + json.dumps(devices, indent=2)
                            )

                        report["chromeDevices"] = devices
                        report["browserVersion"] = browser.version

                    except Exception:
                        await capture_gpu_diagnostics(browser, page, work)
                        raise

                    finally:
                        with contextlib.suppress(Exception):
                            await page.close()

                    (work / "gpu-report.json").write_text(
                        json.dumps(report, indent=2)
                    )
                    print("\nPASS: Chromium WebGPU compute verified.")
                    print(json.dumps(report, indent=2))

                    ws_path = urlsplit(
                        version["webSocketDebuggerUrl"]
                    ).path

                    @web.middleware
                    async def authenticate(request, handler):
                        supplied = request.headers.get("Authorization", "")
                        if not secrets.compare_digest(
                            supplied, "Bearer " + token
                        ):
                            raise web.HTTPUnauthorized()
                        return await handler(request)

                    async def relay(request):
                        async with http.ws_connect(
                            "http://127.0.0.1:9222" + ws_path,
                            max_msg_size=0,
                            heartbeat=30,
                        ) as upstream:
                            downstream = web.WebSocketResponse(
                                max_msg_size=0, heartbeat=30
                            )
                            await downstream.prepare(request)

                            async def pump(source, destination):
                                async for message in source:
                                    if message.type == WSMsgType.TEXT:
                                        await destination.send_str(message.data)
                                    elif message.type == WSMsgType.BINARY:
                                        await destination.send_bytes(message.data)
                                    elif message.type == WSMsgType.ERROR:
                                        break

                            tasks = [
                                asyncio.create_task(pump(downstream, upstream)),
                                asyncio.create_task(pump(upstream, downstream)),
                            ]
                            try:
                                done, _ = await asyncio.wait(
                                    tasks,
                                    return_when=asyncio.FIRST_COMPLETED,
                                )
                                for task in done:
                                    task.result()
                            finally:
                                for task in tasks:
                                    task.cancel()
                                await asyncio.gather(
                                    *tasks, return_exceptions=True
                                )
                                await downstream.close()

                            return downstream

                    app = web.Application(middlewares=[authenticate])
                    app.router.add_get(ws_path, relay)

                    runner = web.AppRunner(
                        app, access_log=None, shutdown_timeout=5
                    )
                    await runner.setup()
                    await web.TCPSite(runner, "127.0.0.1", 8787).start()

                    tunnel_log = open(work / "tunnel.log", "wb")
                    logs.append(tunnel_log)

                    tunnel = await asyncio.create_subprocess_exec(
                        str(cloudflared),
                        "tunnel",
                        "--url", "http://127.0.0.1:8787",
                        "--protocol", "http2",
                        "--no-autoupdate",
                        stdout=tunnel_log,
                        stderr=tunnel_log,
                    )
                    children.append(tunnel)

                    public_url = None
                    for _ in range(120):
                        if tunnel.returncode is not None:
                            raise RuntimeError(
                                "Tunnel exited. Inspect " + str(work / "tunnel.log")
                            )

                        text = (work / "tunnel.log").read_text(errors="replace")
                        match = re.search(
                            r"https://[a-z0-9-]+\.trycloudflare\.com",
                            text,
                        )
                        if match:
                            public_url = match.group()
                            break

                        await asyncio.sleep(0.5)

                    if public_url is None:
                        raise RuntimeError(
                            "Tunnel URL unavailable. Inspect tunnel.log"
                        )

                    endpoint = public_url.replace("https:", "wss:") + ws_path

                    async def check_public_connection():
                        async with http.ws_connect(
                            endpoint,
                            headers={"Authorization": "Bearer " + token},
                            max_msg_size=0,
                        ) as probe:
                            await probe.send_json({
                                "id": 1,
                                "method": "Browser.getVersion",
                            })
                            reply = await probe.receive_json(timeout=10)
                            if reply.get("id") != 1 or "result" not in reply:
                                raise RuntimeError("Unexpected CDP response")

                    public_ready = False
                    last_error = None

                    for _ in range(12):
                        try:
                            await asyncio.wait_for(
                                check_public_connection(),
                                timeout=15,
                            )
                            public_ready = True
                            break
                        except Exception as error:
                            # Avoid printing connection URLs or credentials.
                            last_error = type(error).__name__
                            await asyncio.sleep(2)

                    if not public_ready:
                        raise RuntimeError(
                            "Public CDP verification failed: "
                            + str(last_error)
                            + ". Inspect tunnel.log."
                        )

                    print("\nPASS: Public authenticated CDP connection verified.")
                    print("\n========== PRIVATE CONNECTION DETAILS ==========")
                    print("KAGGLE_CDP_URL=" + endpoint)
                    print("KAGGLE_CDP_TOKEN=" + token)
                    print("================================================")
                    print(f"\nRunning for up to {minutes} minutes.")
                    print("Leave this cell running. Interrupt it to stop.")
                    print("Clear outputs before sharing this notebook.")

                    deadline = time.monotonic() + minutes * 60
                    while time.monotonic() < deadline:
                        if any(
                            child.returncode is not None
                            for child in children
                        ):
                            raise RuntimeError(
                                "Chrome or tunnel stopped. Inspect logs in "
                                + str(work)
                            )
                        await asyncio.sleep(2)

                finally:
                    if runner is not None:
                        await runner.cleanup()

    finally:
        for child in reversed(children):
            if child.returncode is None:
                with contextlib.suppress(ProcessLookupError):
                    child.terminate()
                try:
                    await asyncio.wait_for(child.wait(), timeout=10)
                except asyncio.TimeoutError:
                    with contextlib.suppress(ProcessLookupError):
                        child.kill()
                    await child.wait()

        for log in logs:
            log.close()

        profile.cleanup()
        print("Remote browser and tunnel stopped.")


await remote_browser(minutes=120)